# 🗄️ Curiosity On Demand - Database Explorer

This notebook allows you to inspect, query, and manage the SQLite database backing the **Curiosity On Demand** pipeline.

### Features:
- 📋 **List Projects:** View all research project sessions and lifecycle states.
- 🔍 **Inspect Artifacts:** Parse and display stored JSON artifacts (`ResearchReport`, `Script`, `AnimationStyle`, `AnimationStoryboard`).
- 🖼️ **Media Assets:** Check paths for rendered sample frame snapshots (PNG) and master MP4 videos.

## 1. Setup & Database Connection

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import select

# Ensure project root is in sys.path when running from notebooks/
project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables (.env)
load_dotenv(project_root / ".env")

from config.settings import get_settings
from core.database import DatabaseManager, ProjectRecord, get_db_manager
from core.models import ProjectSession

settings = get_settings()
db: DatabaseManager = get_db_manager()

# Ensure database tables exist
await db.init_db()

print("✅ Connected to SQLite database.")
print(f"• DB Path: {settings.sqlite_db_path.resolve()}")

## 2. List All Project Sessions
Query the database for all recorded project sessions.

In [ ]:
async with db.session_factory() as session:
    stmt = select(ProjectRecord).order_by(ProjectRecord.created_at.desc())
    result = await session.execute(stmt)
    records = result.scalars().all()

print(f"📊 Total Projects in Database: {len(records)}\n")
for r in records:
    print(f"ID:        {r.project_id}")
    print(f"Topic:     {r.topic}")
    print(f"State:     {r.state}")
    print(f"Chat ID:   {r.chat_id} (User: {r.user_id})")
    print(f"Created:   {r.created_at} | Updated: {r.updated_at}")
    print("-" * 60)

## 3. Inspect Specific Project Artifacts
Inspect the strongly-typed `ProjectSession` model and its research report, narration script, animation style, and video paths.

In [ ]:
if records:
    target_project_id = records[0].project_id
    project: ProjectSession | None = await db.get_project(target_project_id)

    if project:
        print(f"🎯 Project ID: {project.project_id}")
        print(f"Topic:        {project.topic}")
        print(f"State:        {project.state.value}")
        if project.research_report:
            print(f"\n💡 Research Summary:\n{project.research_report.summary}")
            print(
                f"\n📚 Researched Subtopics ({len(project.research_report.sections)} sections):"
            )
            for i, sec in enumerate(project.research_report.sections, 1):
                print(f"  {i}. {sec.topic}: {sec.content[:100]}...")
        if project.script:
            print(f"\n🎬 Script ({len(project.script.scenes)} scenes):")
            for s in project.script.scenes:
                print(f"  • Scene {s.scene_number} [{s.title}]:")
                print(f"    - Summary: {s.summary}")
                print(f"    - Visuals: {s.description}")
                print(f"    - Narration: {s.narration[:80]}...")
        if project.style:
            print(f"\n🎨 Visual Style: {project.style.style_name}")
            print(f"  • Palette: {', '.join(project.style.color_palette)}")
            print(f"  • Sample Frame: {project.sample_frame_path}")
        if project.final_video_path:
            print(f"\n🎞️ Final Video: {project.final_video_path}")
else:
    print("ℹ️ No projects in database yet.")